# AML Baseline 1 — Stage 2: C4 + D1 Full-data Fine-tuning (10 Epochs) + Final Official Evaluation

**Single Google Colab Notebook — no external `.py` file needed.**

- Crop: `C4_dropout_03_P15/best.pt`, Disease: `D1_reference_P15/best.pt`.
- Re-train Stage 2 **from original Stage 1 best checkpoints**, not from the older 5-epoch Stage 2 model.
- 8,000 original training images, low LR (backbone `1e-6`, classifier `1e-4`), AdamW, cosine annealing `T_max=10`.
- Train Crop and Disease separately for **10 epochs maximum**. After training, select each model's **highest recorded training Top-1 accuracy** (earliest epoch on an exact tie), following the agreed teacher-guided method. A separate plateau diagnostic is included; 10 epochs are an observation limit, not a guarantee of plateau.
- **Freeze the two selected checkpoints before enabling Official Test.** Never use Official Test to select epochs or modify training.
- Official Test: Seen 2,000, Unseen 1,448; all 9,625 crop–disease candidates. Save Top-1/Top-5/H, full predictions, confusion matrices and misclassification galleries.

**Methodology caveat:** Highest training accuracy does not necessarily give best generalisation; report this as *training-accuracy-based checkpoint selection*, not validation-based early stopping. The former 1,095 Development Validation images are part of the Stage 2 training set and are not independent validation here.

**Folder safety:** This notebook writes only into `AML_B1_Clean_Results/fine_tuning/Stage2_C4_D1_Full8000_10Epochs/`. The earlier `...5Epochs` directory is untouched.


## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Configuration and environment

In [ ]:
from pathlib import Path
import os, json, random, time, shutil, subprocess, copy, gc, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DRIVE_PROJECT = Path('/content/drive/MyDrive/DPD-Small')
RESULTS = Path('/content/drive/MyDrive/AML_B1_Clean_Results')
LOCAL_PROJECT = Path('/content/DPD-Small')
RUN_NAME = 'Stage2_C4_D1_Full8000_10Epochs'
OUT_ROOT = RESULTS / 'fine_tuning' / RUN_NAME

CROP_RUN = 'C4_dropout_03_P15'
DISEASE_RUN = 'D1_reference_P15'
CROP_CKPT = RESULTS / 'experiments' / 'crop' / CROP_RUN / 'best.pt'
DISEASE_CKPT = RESULTS / 'experiments' / 'disease' / DISEASE_RUN / 'best.pt'

FIXED_EPOCHS = 10
BACKBONE_LR = 1e-6
HEAD_LR = 1e-4
BATCH_SIZE = 32
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Please enable GPU: Runtime > Change runtime type > GPU'
AMP = DEVICE.type == 'cuda'
assert DRIVE_PROJECT.is_dir(), f'Missing dataset folder: {DRIVE_PROJECT}'
assert CROP_CKPT.is_file(), f'Missing Stage 1 Crop checkpoint: {CROP_CKPT}'
assert DISEASE_CKPT.is_file(), f'Missing Stage 1 Disease checkpoint: {DISEASE_CKPT}'

# Do not create the run folders until the input checks pass.
print('GPU:', torch.cuda.get_device_name(0))
print('Crop checkpoint:', CROP_CKPT)
print('Disease checkpoint:', DISEASE_CKPT)
print('Output root:', OUT_ROOT)
print('Policy: 8,000 original training images / 10 epochs + training-accuracy checkpoint selection / no validation or official test')

## Cell 3 — Copy original training CSV and images to Colab local disk
Copying images locally improves data-loader throughput; `rsync --size-only` can resume a partly completed transfer. This cell **does not** read or copy official test CSVs.

In [ ]:
LOCAL_PROJECT.mkdir(parents=True, exist_ok=True)

def find_child(parent, name):
    hits = [x for x in parent.iterdir() if x.name.lower() == name.lower()]
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected exactly one {name!r} inside {parent}; found {hits}')
    return hits[0]

source_images = find_child(DRIVE_PROJECT, 'images')
local_images = LOCAL_PROJECT / 'images'
local_images.mkdir(parents=True, exist_ok=True)
print('Copying/resuming images from Drive to local Colab...')
subprocess.run([
    'rsync', '-rt', '--size-only', '--info=progress2',
    str(source_images) + '/', str(local_images) + '/'
], check=True)

source_train_csv = find_child(DRIVE_PROJECT, 'DPD_Small_train.csv')
shutil.copy2(source_train_csv, LOCAL_PROJECT / source_train_csv.name)
print('Original train CSV:', source_train_csv.name)
print('Local images:', local_images)

## Cell 4 — Read exactly 8,000 original training records and verify class mapping
The class-order mapping is taken from the **Stage 1 checkpoints**, not inferred afresh from an arbitrary split. This prevents silent label remapping.

In [ ]:
COLUMNS = ['filename','crop_id','disease_id','composition_id',
           'crop_name','disease_name','composition_name']

def read_original_csv(path):
    with open(path, 'r', encoding='utf-8-sig') as file:
        has_header = file.readline().strip().split(',')[0].strip().lower() == 'filename'
    df = pd.read_csv(path, names=COLUMNS, header=0 if has_header else None, encoding='utf-8-sig')
    for col in ['crop_id','disease_id','composition_id']:
        df[col] = pd.to_numeric(df[col], errors='raise').astype(int)
    df['filename'] = df['filename'].astype(str)
    return df

full_train_df = read_original_csv(LOCAL_PROJECT / source_train_csv.name)
assert len(full_train_df) == 8000, f'Expected exactly 8000, got {len(full_train_df)}'
assert full_train_df.filename.is_unique, 'Found duplicate training filename(s)'

# Checkpoint metadata is trusted only after explicit consistency checks.
def read_checkpoint(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    assert isinstance(ck, dict) and 'state_dict' in ck, f'Unexpected checkpoint format: {path}'
    return ck

crop_ck = read_checkpoint(CROP_CKPT)
disease_ck = read_checkpoint(DISEASE_CKPT)
assert crop_ck.get('task') == 'crop' and disease_ck.get('task') == 'disease'
crop_ids = [int(v) for v in crop_ck['crop_ids']]
disease_ids = [int(v) for v in crop_ck['disease_ids']]
assert len(crop_ids) == 55 and len(disease_ids) == 175
assert crop_ids == [int(v) for v in disease_ck['crop_ids']]
assert disease_ids == [int(v) for v in disease_ck['disease_ids']]
assert set(full_train_df.crop_id) == set(crop_ids)
assert set(full_train_df.disease_id) == set(disease_ids)

crop_to_idx = {v:i for i,v in enumerate(crop_ids)}
disease_to_idx = {v:i for i,v in enumerate(disease_ids)}
for ck, task, expected_dropout in [(crop_ck,'crop',0.3),(disease_ck,'disease',0.2)]:
    actual = float(ck.get('config',{}).get('dropout',-1))
    assert abs(actual - expected_dropout) < 1e-8, f'{task}: checkpoint dropout {actual} != {expected_dropout}'
    print(f'{task.upper():7s} source epoch: {ck.get("epoch")}; dropout: {actual}')

# Original E1/E4 checkpoint class mapping should match this file as well.
mapping_file = RESULTS / 'class_mapping.json'
if mapping_file.is_file():
    m = json.loads(mapping_file.read_text())
    assert [int(x) for x in m['crop_ids']] == crop_ids
    assert [int(x) for x in m['disease_ids']] == disease_ids

print(f'PASS: {len(full_train_df):,} unique training images, {len(crop_ids)} crops, {len(disease_ids)} diseases')
print('All labels are mapped consistently with both Stage 1 checkpoints.')

## Cell 5 — Index images and preserve original E1–E4 augmentation

In [ ]:
image_lookup = {}
for f in tqdm(local_images.rglob('*'), desc='Indexing local images'):
    if f.is_file() and f.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
        if f.name in image_lookup:
            raise RuntimeError(f'Ambiguous duplicate image basename: {f.name}')
        image_lookup[f.name] = f
missing = set(full_train_df.filename) - set(image_lookup)
assert not missing, f'Missing {len(missing)} training images, e.g.: {list(sorted(missing))[:5]}'

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class PlantDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(image_lookup[row.filename]) as im:
            x = train_transform(im.convert('RGB'))
        return x, crop_to_idx[int(row.crop_id)], disease_to_idx[int(row.disease_id)]

print('All 8,000 local image paths verified.')
print('Train augmentation matches E1–E4; no new augmentation experiment is introduced.')

## Cell 6 — Model, optimizer, training and safe resume helpers

In [ ]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build_model(task, dropout):
    # Must match Stage 1 architecture exactly. No ImageNet download needed.
    model = models.efficientnet_b0(weights=None)
    dim = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(float(dropout)),
        nn.Linear(dim, 55 if task == 'crop' else 175),
    )
    return model.to(DEVICE)

def optimizer_for(model):
    return torch.optim.AdamW([
        {'params': model.features.parameters(), 'lr': BACKBONE_LR},
        {'params': model.classifier.parameters(), 'lr': HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)

def atomic_torch_save(data, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + '.tmp')
    try:
        torch.save(data, temporary)
        os.replace(temporary, target)
    finally:
        if temporary.exists(): temporary.unlink()

def atomic_csv_save(df, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_name(target.name + '.tmp')
    try:
        df.to_csv(tmp, index=False)
        os.replace(tmp, target)
    finally:
        if tmp.exists(): tmp.unlink()

def rng_state():
    state = {'python':random.getstate(), 'numpy':np.random.get_state(),
             'torch':torch.get_rng_state()}
    if DEVICE.type == 'cuda': state['cuda'] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    if DEVICE.type == 'cuda' and 'cuda' in state:
        torch.cuda.set_rng_state_all(state['cuda'])

CONFIG = {
    'run_name': RUN_NAME,
    'source_crop': CROP_RUN,
    'source_disease': DISEASE_RUN,
    'source_crop_checkpoint': str(CROP_CKPT),
    'source_disease_checkpoint': str(DISEASE_CKPT),
    'training_image_count': 8000,
    'epochs': FIXED_EPOCHS,
    'batch_size': BATCH_SIZE,
    'backbone_lr': BACKBONE_LR,
    'head_lr': HEAD_LR,
    'weight_decay': WEIGHT_DECAY,
    'crop_dropout': 0.3,
    'disease_dropout': 0.2,
    'seed': SEED,
    'scheduler': 'CosineAnnealingLR(T_max=10)',
    'validation': 'DISABLED: stage 2 uses all original 8000 training images',
    'official_test': 'NOT USED for checkpoint selection',
    'checkpoint_selection': 'Max epoch-level training Top-1 accuracy (earliest tie) after 10 epochs',
    'crop_ids': crop_ids,
    'disease_ids': disease_ids,
    'augmentations': ['RandomResizedCrop(224,scale=(0.85,1.0))',
                      'RandomHorizontalFlip(p=0.5)', 'RandomRotation(10)',
                      'ColorJitter(brightness=0.15,contrast=0.15)',
                      'ToTensor()', 'ImageNetNormalize'],
}
CONFIG_SIGNATURE = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
print('Resume helpers ready. Config signature:', CONFIG_SIGNATURE[:16])

## Cell 7 — Ten-epoch training loop, checkpoints, graphs and automatic resume

Each task saves `epoch_01.pt` through `epoch_10.pt`, optimizer/scheduler/scaler state in `resume_latest.pt`, `history.csv`, `run_config.json`, `COMPLETE.json`, and live Loss / Accuracy graphs. `T_max=10` starts fresh from Stage 1 weights; **do not** continue the previous 5-epoch run.

The 10-epoch run is a budget; checkpoint selection happens only in **Cell 10A**, using *training accuracy*, before any Official Test.

In [ ]:
def plot_task_history(task):
    out = OUT_ROOT / task
    file = out / 'history.csv'
    if not file.is_file(): return
    df = pd.read_csv(file)
    if df.empty: return
    gd = out / 'graphs'
    gd.mkdir(parents=True, exist_ok=True)
    charts = [
        ('training_loss.png', 'Training Cross-Entropy Loss', 'train_loss', 'Loss', 1),
        ('training_accuracy.png', 'Training Top-1 Accuracy', 'train_acc', 'Accuracy (%)', 100),
    ]
    for filename, title, col, ylabel, factor in charts:
        fig, ax = plt.subplots(figsize=(7,4))
        ax.plot(df['epoch'], df[col] * factor, marker='o', linewidth=2)
        ax.set(xticks=list(df.epoch), xlabel='Stage 2 Fine-tuning Epoch', ylabel=ylabel,
               title=f'{task.upper()} — {title}')
        ax.grid(alpha=.25)
        fig.tight_layout()
        fig.savefig(gd / filename, dpi=175)
        plt.show()
        plt.close(fig)
    print('Graphs saved to:', gd)

def train_stage2(task, source_ckpt):
    assert task in ('crop', 'disease')
    seed_all(SEED)
    out = OUT_ROOT / task
    out.mkdir(parents=True, exist_ok=True)
    config_file = out / 'run_config.json'
    complete_file = out / 'COMPLETE.json'
    resume_file = out / 'resume_latest.pt'
    history_file = out / 'history.csv'

    if config_file.exists():
        stored = json.loads(config_file.read_text())
        assert stored.get('signature') == CONFIG_SIGNATURE, (
            f'{task} output belongs to a different configuration. Do not overwrite: {out}'
        )
        assert stored.get('source_checkpoint') == str(source_ckpt)
    else:
        # Never silently merge partial output with a fresh run.
        if any(out.iterdir()):
            raise RuntimeError(f'Existing output without run_config.json: {out}. Inspect or rename it first.')
        config_file.write_text(json.dumps({
            'signature':CONFIG_SIGNATURE,
            'source_checkpoint':str(source_ckpt),
            **CONFIG
        }, indent=2))

    if complete_file.is_file():
        assert (out / f'epoch_{FIXED_EPOCHS:02d}.pt').is_file()
        print(f'{task.upper()} already COMPLETE. Skipping: {out}')
        plot_task_history(task)
        return

    ck = read_checkpoint(source_ckpt)
    dropout = float(ck['config']['dropout'])
    model = build_model(task, dropout)
    model.load_state_dict(ck['state_dict'], strict=True)
    optimizer = optimizer_for(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FIXED_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)

    history = []
    start_epoch = 1
    if resume_file.is_file():
        saved = torch.load(resume_file, map_location='cpu', weights_only=False)
        assert saved['signature'] == CONFIG_SIGNATURE
        assert saved['task'] == task
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        history = saved['history']
        start_epoch = int(saved['finished_epoch']) + 1
        restore_rng_state(saved['rng'])
        print(f'{task.upper()} resuming from finished epoch {start_epoch-1}')
    elif history_file.exists():
        raise RuntimeError(f'{task}: history exists but full resume checkpoint missing. Not safe to restart.')
    else:
        print(f'{task.upper()} starting Stage 2 from Stage 1 best checkpoint epoch {ck.get("epoch")}')

    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED + (0 if task=='crop' else 100))
    # Training resumes at epoch boundaries; shuffle is varied deterministically per epoch.
    loader = DataLoader(PlantDataset(full_train_df), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=(NUM_WORKERS > 0), generator=loader_generator)
    loss_function = nn.CrossEntropyLoss()
    try:
        for epoch in range(start_epoch, FIXED_EPOCHS + 1):
            # Reset generator per epoch, so a resumed epoch gets the same sample order.
            loader_generator.manual_seed(SEED + epoch * 101 + (0 if task=='crop' else 1))
            model.train()
            t0=time.time()
            total, correct, loss_total = 0, 0, 0.0
            bar = tqdm(loader, desc=f'STAGE 2 {task.upper()} epoch {epoch}/{FIXED_EPOCHS}', leave=False)
            for x, yc, yd in bar:
                x = x.to(DEVICE, non_blocking=True)
                y = (yc if task=='crop' else yd).to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast('cuda', enabled=AMP):
                    logits = model(x)
                    loss = loss_function(logits, y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                batch_n = y.size(0)
                total += batch_n
                correct += (logits.argmax(dim=1)==y).sum().item()
                loss_total += loss.detach().item() * batch_n
                bar.set_postfix(loss=f'{loss_total/total:.4f}', accuracy=f'{correct/total:.2%}')
            assert total == 8000, f'Expected 8000 samples in epoch {epoch}, got {total}'
            scheduler.step()
            row = {'epoch':epoch, 'training_images':total,
                   'train_loss':loss_total/total, 'train_acc':correct/total,
                   'seconds':time.time()-t0}
            history.append(row)
            # Original Stage 1-compatible epoch checkpoint format.
            epoch_ckpt = {
                'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'task':task,'epoch':epoch,
                'config': {'backbone_lr':BACKBONE_LR,'head_lr':HEAD_LR,'batch_size':BATCH_SIZE,
                           'epochs':FIXED_EPOCHS,'weight_decay':WEIGHT_DECAY,'dropout':dropout},
                'crop_ids':crop_ids,'disease_ids':disease_ids,'seed':SEED,
                'source_run':CROP_RUN if task=='crop' else DISEASE_RUN,
                'stage':'stage2_full_data_fine_tuning', 'selection_metric':None,
            }
            atomic_torch_save(epoch_ckpt, out / f'epoch_{epoch:02d}.pt')
            atomic_torch_save({
                'signature':CONFIG_SIGNATURE,'task':task,'finished_epoch':epoch,
                'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(), 'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(), 'history':history, 'rng':rng_state()
            }, resume_file)
            atomic_csv_save(pd.DataFrame(history), history_file)
            print(f'{task.upper()} | epoch {epoch:02d}/{FIXED_EPOCHS} | '
                  f'Train Loss {row["train_loss"]:.4f} | '
                  f'Train Top-1 {row["train_acc"]:.2%} | '
                  f'Time {row["seconds"]:.0f}s | checkpoint saved', flush=True)
            plot_task_history(task)

        assert len(history) == FIXED_EPOCHS
        complete_file.write_text(json.dumps({
            'task':task, 'source_checkpoint':str(source_ckpt),
            'final_epoch':FIXED_EPOCHS, 'training_image_count':8000,
            'final_checkpoint':str(out / f'epoch_{FIXED_EPOCHS:02d}.pt'),
            'selection_policy':'Highest training Top-1 accuracy over 10 epochs; earliest tie; no validation/test selection',
            'signature':CONFIG_SIGNATURE,
        }, indent=2))
        print(f'COMPLETE — {task.upper()}: {out / f"epoch_{FIXED_EPOCHS:02d}.pt"}')
    finally:
        del model, optimizer, scheduler, scaler, loader
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print('Training and plotting functions defined. No training has started.')

## Cell 8 — Re-fine-tune **C4 Crop CNN** (10 Epochs maximum)

New output folder. Resume from `resume_latest.pt` if this 10-epoch Colab run disconnects. Does not reuse or overwrite the old 5-epoch Stage 2 run.

In [ ]:
train_stage2('crop', CROP_CKPT)

## Cell 9 — Re-fine-tune **D1 Disease CNN** (10 Epochs maximum)

Disease trains independently and may select a different highest-training-accuracy epoch from Crop.

In [ ]:
train_stage2('disease', DISEASE_CKPT)

## Cell 10 — Fine-tuning Completion Summary

This table shows training results and the best training-accuracy epoch *observed*. Then run Cell 10A to freeze the selected checkpoints. **Do not start Official Test before Cell 10A.**

In [ ]:
rows = []
for task in ['crop','disease']:
    out = OUT_ROOT / task
    hist = out / 'history.csv'
    done = (out / 'COMPLETE.json').exists()
    if not hist.exists():
        rows.append({'Task':task.upper(),'Status':'NOT STARTED'})
        continue
    df = pd.read_csv(hist)
    rows.append({
        'Task':task.upper(),
        'Source Stage 1':CROP_RUN if task=='crop' else DISEASE_RUN,
        'Status':'COMPLETE' if done else 'INCOMPLETE / RESUMABLE',
        'Epochs Finished':len(df),
        'Training Loss (Last Epoch)':round(float(df.train_loss.iloc[-1]),4),
        'Training Accuracy (Last Epoch, %)':round(float(df.train_acc.iloc[-1])*100,2),
        'Best Observed Training Accuracy (%)':round(float(df.train_acc.max())*100,2),
        'Best Observed Epoch':int(df.loc[df.train_acc.idxmax(),'epoch']),
    })
summary = pd.DataFrame(rows)
display(summary)
OUT_ROOT.mkdir(parents=True,exist_ok=True)
atomic_csv_save(summary, OUT_ROOT / 'stage2_run_summary.csv')
if all((OUT_ROOT / x / 'COMPLETE.json').exists() for x in ['crop','disease']):
    print('BOTH MODELS FINISHED 10 EPOCHS. Proceed to Cell 10A to freeze checkpoint selection BEFORE Official Test.')
else:
    print('At least one model is incomplete. Re-run its training cell to resume before selection.')


## Cell 10A — Training Accuracy Plateau Review & Final Checkpoint Selection (NO TEST DATA)

**Run only after both 10-epoch tasks finish.** This is the checkpoint-selection rule discussed with your lecturer:

1. Inspect each task's training accuracy curve and a 3-epoch plateau diagnostic (range ≤ **0.10 percentage points**). This is a descriptive indicator only; don't claim a plateau if it is not met.
2. Select **the epoch with the maximum measured Training Top-1 Accuracy among epochs 1–10** separately for Crop and Disease. On an exact tie choose the earliest.
3. Copy the existing saved epoch checkpoint to `best_training_accuracy.pt`; write a CSV/JSON manifest recording the epoch and rule.
4. **The choice is locked before Official Test** and cannot be revised based on Official Test results. Do not claim Training Accuracy is an independent generalisation metric.

If the curves still rise sharply at Epoch 10, report that the maximum budget was reached without confirmed plateau. Do not use Official Test to decide whether to add more epochs.

In [ ]:
# =============================================================
# STAGE 2 CHECKPOINT SELECTION — TRAINING ACCURACY ONLY
# No Development Validation and NO Official Test are accessed.
# =============================================================
import shutil

PLATEAU_WINDOW = 3
PLATEAU_RANGE_PP = 0.10   # descriptive only, 0.10 percentage points
selection_rows = []
for task in ['crop', 'disease']:
    folder = OUT_ROOT / task
    assert (folder / 'COMPLETE.json').is_file(), f'{task}: finish all 10 epochs before selecting'
    hist_path = folder / 'history.csv'
    h = pd.read_csv(hist_path).sort_values('epoch').reset_index(drop=True)
    assert len(h) == FIXED_EPOCHS == 10 and h.epoch.tolist() == list(range(1, 11))
    assert np.isfinite(h[['train_acc','train_loss']].to_numpy()).all()
    # Higher measured training accuracy wins; earliest epoch in the case of exact equality.
    best_idx = int(np.argmax(h.train_acc.to_numpy()))
    best_epoch = int(h.loc[best_idx, 'epoch'])
    best_acc = float(h.loc[best_idx, 'train_acc'])
    recent = h.tail(PLATEAU_WINDOW).train_acc.astype(float) * 100
    range_pp = float(recent.max() - recent.min())
    plateau = (range_pp <= PLATEAU_RANGE_PP)
    source = folder / f'epoch_{best_epoch:02d}.pt'
    selected = folder / 'best_training_accuracy.pt'
    assert source.is_file(), source
    # Safe replay: do not silently swap a locked checkpoint or overwrite a frozen Official Test.
    previous = folder / 'training_accuracy_selection.json'
    record = {
        'task':task, 'max_epochs':FIXED_EPOCHS,
        'selected_epoch':best_epoch,
        'selected_training_accuracy_percent':round(best_acc*100,6),
        'selection_rule':'maximum recorded epoch-level training Top-1 accuracy, earliest exact tie',
        'plateau_window':PLATEAU_WINDOW,
        'plateau_range_threshold_percentage_points':PLATEAU_RANGE_PP,
        'last_window_accuracy_range_percentage_points':round(range_pp,6),
        'plateau_indicator':bool(plateau),
        'epoch_source_checkpoint':str(source),
        'selected_checkpoint':str(selected),
        'warning':'Training accuracy is not an independent estimate of test generalisation.'
    }
    if previous.is_file():
        assert json.loads(previous.read_text()) == record, f'Previously frozen selection differs for {task}'
        assert selected.is_file(), selected
    else:
        assert not (OUT_ROOT / 'official_test_final' / 'frozen_evaluation_protocol.json').exists(), 'Official Test was already frozen'
        shutil.copy2(source, selected)
        previous.write_text(json.dumps(record, indent=2))
    selection_rows.append({
        'Task':task.upper(), 'Epochs Completed':len(h),
        'Selected Epoch':best_epoch,
        'Highest Training Accuracy (%)':round(100*best_acc,3),
        'Training Loss at Selected Epoch':round(float(h.loc[best_idx,'train_loss']),4),
        'Last 3 Epochs Range (pp)':round(range_pp,3),
        'Plateau Indicator (0.10 pp)': 'YES' if plateau else 'NOT CONFIRMED',
        'Final Checkpoint':str(selected),
    })

selection_df = pd.DataFrame(selection_rows)
atomic_csv_save(selection_df, OUT_ROOT / '10epoch_training_accuracy_checkpoint_selection.csv')
print('FINAL STAGE 2 TRAINING-ACCURACY CHECKPOINT SELECTION (no Official Test)')
display(selection_df)
for task in ['crop','disease']:
    plot_task_history(task)
print('Frozen selection saved under:', OUT_ROOT)
print('Selection is based only on training performance, not generalisation.')


---
# Part B — FINAL OFFICIAL TEST (run only after selecting final checkpoints)

This section tests **Official Seen (2,000)** and **Official Unseen (1,448)**, reports independent Crop/Disease Top-1/Top-5 and **Composition Top-1/Top-5 / Seen–Unseen Harmonic Mean** using all **9,625 crop–disease candidate pairs**, and saves complete predictions, confusion matrices and per-category misclassification galleries.

**Freeze checkpoints and the selection rule before enabling evaluation.** The Official Test must not be used to compare Stage 2 epochs, tune learning rate/augmentation, or switch models. Default checkpoint selection is the **selected training-accuracy epoch** from Part A. If using a teacher-approved training-accuracy plateau protocol instead, finish that selection solely on training data and provide its two frozen paths below *before* running the test.

Part B depends on setup Cells 1–6 (mount Drive, copy/index data, labels, build model) but **does not need you to re-run training Cells 8–9**.

## Cell 11 — Confirm Frozen Selected Checkpoints, then enable Official Test

**Complete Cell 10A first.** The script automatically loads `crop/best_training_accuracy.pt` and `disease/best_training_accuracy.pt`, and verifies both selection manifests. The Official Test must not influence this selection.

Set `RUN_FINAL_OFFICIAL_TEST=True` only after both models have completed 10 epochs, selection has been frozen, and you are ready for the one-time final evaluation.

In [ ]:
# -------------------------------------------------------------
# FINAL OFFICIAL TEST — INPUTS / SAFETY CHECKS
# -------------------------------------------------------------
RUN_FINAL_OFFICIAL_TEST = True  # Change to True only after FINAL checkpoint selection.

# By default use 10-epoch fine-tuning outputs. Never auto-pick based on Official Test.
FINAL_CROP_CHECKPOINT = OUT_ROOT / 'crop' / 'best_training_accuracy.pt'
FINAL_DISEASE_CHECKPOINT = OUT_ROOT / 'disease' / 'best_training_accuracy.pt'
FINAL_SELECTION_POLICY = 'highest_training_top1_accuracy_per_task_from_10_epochs_earliest_tie_frozen_before_official_test'

OFFICIAL_DIR = OUT_ROOT / 'official_test_final'
MAX_GALLERY_PER_CATEGORY = 16
EVAL_BATCH_SIZE = 32

if not RUN_FINAL_OFFICIAL_TEST:
    print('OFFICIAL TEST DISABLED. First freeze the final crop/disease checkpoints; then set RUN_FINAL_OFFICIAL_TEST=True.')
else:
    assert (OUT_ROOT / 'crop' / 'training_accuracy_selection.json').is_file(), 'Run Cell 10A first'
    assert (OUT_ROOT / 'disease' / 'training_accuracy_selection.json').is_file(), 'Run Cell 10A first'
    assert FINAL_CROP_CHECKPOINT.is_file(), FINAL_CROP_CHECKPOINT
    assert FINAL_DISEASE_CHECKPOINT.is_file(), FINAL_DISEASE_CHECKPOINT
    for task, checkpoint in [('crop',FINAL_CROP_CHECKPOINT),('disease',FINAL_DISEASE_CHECKPOINT)]:
        selection = json.loads((OUT_ROOT / task / 'training_accuracy_selection.json').read_text())
        assert selection['selected_checkpoint'] == str(checkpoint)
        assert (OUT_ROOT / task / 'COMPLETE.json').is_file(), f'{task}: training incomplete'
    official_seen_csv = find_child(DRIVE_PROJECT, 'DPD_Small_test_seen.csv')
    official_unseen_csv = find_child(DRIVE_PROJECT, 'DPD_Small_test_unseen.csv')
    official_dfs = {
        'seen': read_original_csv(official_seen_csv),
        'unseen': read_original_csv(official_unseen_csv),
    }
    assert len(official_dfs['seen']) == 2000
    assert len(official_dfs['unseen']) == 1448
    assert set(official_dfs['seen'].filename).isdisjoint(set(official_dfs['unseen'].filename))
    assert not (set(full_train_df.filename) & (set(official_dfs['seen'].filename)|set(official_dfs['unseen'].filename)))
    for split, data in official_dfs.items():
        assert data.filename.is_unique
        assert set(data.crop_id).issubset(crop_to_idx)
        assert set(data.disease_id).issubset(disease_to_idx)
        missing = set(data.filename) - set(image_lookup)
        if missing:
            # Earlier setup indexes local train images. If official images are in Drive but not local, copy these files.
            for name in tqdm(sorted(missing), desc=f'Copying {split} test images'):
                hits = list(source_images.rglob(name))
                assert len(hits)==1, f'{name}: expected one source image, found {len(hits)}'
                dst = local_images / name
                shutil.copy2(hits[0], dst)
                image_lookup[name] = dst
        assert all(Path(image_lookup[x]).is_file() for x in data.filename)
        print(f'Official {split.upper():6s}: {len(data):,} images, file overlap with training = 0')
    print('Frozen Crop:', FINAL_CROP_CHECKPOINT)
    print('Frozen Disease:', FINAL_DISEASE_CHECKPOINT)
    print('Selection rule:', FINAL_SELECTION_POLICY)
    print('Official output:', OFFICIAL_DIR)

## Cell 12 — Official inference and all evaluation metrics

For all 55 × 175 candidate compositions, score each pair as `log P(crop | image) + log P(disease | image)`. The Top-1 composition equals both separate Top-1 predictions being correct; Composition Top-5 is ranked over the full Cartesian product, **not** the average of individual Top-5 accuracies. No training or validation updates happen here.

In [ ]:
from sklearn.metrics import confusion_matrix

class OfficialTestDataset(Dataset):
    def __init__(self, df):
        self.df=df.reset_index(drop=True)
        self.tf=transforms.Compose([
            transforms.Resize(256), transforms.CenterCrop(224),
            transforms.ToTensor(), transforms.Normalize(MEAN, STD)
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row=self.df.iloc[i]
        with Image.open(image_lookup[row.filename]) as im:
            x=self.tf(im.convert('RGB'))
        return x, crop_to_idx[int(row.crop_id)], disease_to_idx[int(row.disease_id)]

def load_frozen_model(path, task, expected_dropout):
    ck=read_checkpoint(path)
    assert ck.get('task')==task and ck.get('stage')=='stage2_full_data_fine_tuning', (path,ck.get('task'),ck.get('stage'))
    assert [int(x) for x in ck['crop_ids']] == crop_ids
    assert [int(x) for x in ck['disease_ids']] == disease_ids
    assert abs(float(ck['config']['dropout'])-expected_dropout)<1e-8
    model=build_model(task,expected_dropout)
    model.load_state_dict(ck['state_dict'],strict=True)
    model.eval()
    return model

def harmonic(a,b):
    return 0.0 if a+b==0 else (2*a*b/(a+b))

def evaluate_official_split(split, crop_model, disease_model):
    df=official_dfs[split].reset_index(drop=True)
    loader=DataLoader(OfficialTestDataset(df),batch_size=EVAL_BATCH_SIZE,shuffle=False,
                      num_workers=NUM_WORKERS,pin_memory=True)
    cps=[];dps=[]; yc=[];yd=[]
    with torch.inference_mode():
        for x, truth_crop, truth_disease in tqdm(loader,desc=f'Official {split}'):
            x=x.to(DEVICE,non_blocking=True)
            with torch.autocast('cuda',enabled=AMP):
                cl=crop_model(x).float()
                dl=disease_model(x).float()
            cps.append(torch.log_softmax(cl,dim=-1).cpu().numpy())
            dps.append(torch.log_softmax(dl,dim=-1).cpu().numpy())
            yc.append(truth_crop.numpy()); yd.append(truth_disease.numpy())
    cp=np.concatenate(cps); dp=np.concatenate(dps)
    tc=np.concatenate(yc).astype(int); td=np.concatenate(yd).astype(int)
    assert cp.shape==(len(df),55) and dp.shape==(len(df),175)
    assert np.array_equal(tc,df.crop_id.map(crop_to_idx).to_numpy())
    assert np.array_equal(td,df.disease_id.map(disease_to_idx).to_numpy())
    pred_c=cp.argmax(1); pred_d=dp.argmax(1)
    true_pair=tc*175+td
    score=(cp[:,:,None]+dp[:,None,:]).reshape(-1,9625)
    pred_pair=score.argmax(1)
    pair_top5=np.argpartition(score,-5,axis=1)[:,-5:]
    c_top5=np.argpartition(cp,-5,axis=1)[:,-5:]
    d_top5=np.argpartition(dp,-5,axis=1)[:,-5:]
    comp_correct=(pred_pair==true_pair)
    assert np.array_equal(comp_correct,(pred_c==tc)&(pred_d==td)), 'Full-cartesian composition calculation inconsistent'
    stats={
       'crop_top1':np.mean(pred_c==tc),
       'crop_top5':np.mean(np.any(c_top5==tc[:,None],axis=1)),
       'disease_top1':np.mean(pred_d==td),
       'disease_top5':np.mean(np.any(d_top5==td[:,None],axis=1)),
       'composition_top1':np.mean(comp_correct),
       'composition_top5':np.mean(np.any(pair_top5==true_pair[:,None],axis=1)),
    }
    categories=np.select([
        (pred_c==tc)&(pred_d==td),
        (pred_c!=tc)&(pred_d==td),
        (pred_c==tc)&(pred_d!=td),
        (pred_c!=tc)&(pred_d!=td),
    ],['both_correct','crop_wrong_only','disease_wrong_only','both_wrong'],default='unknown')
    out=df.copy()
    out['split']=split
    out['true_crop_idx']=tc;out['true_disease_idx']=td
    out['pred_crop_id']=[crop_ids[x] for x in pred_c]
    out['pred_disease_id']=[disease_ids[x] for x in pred_d]
    out['pred_composition_crop_id']=[crop_ids[x//175] for x in pred_pair]
    out['pred_composition_disease_id']=[disease_ids[x%175] for x in pred_pair]
    out['crop_confidence']=np.exp(cp.max(1));out['disease_confidence']=np.exp(dp.max(1))
    out['crop_correct']=pred_c==tc;out['disease_correct']=pred_d==td
    out['composition_correct']=comp_correct
    out['crop_top5_correct']=np.any(c_top5==tc[:,None],axis=1)
    out['disease_top5_correct']=np.any(d_top5==td[:,None],axis=1)
    out['composition_top5_correct']=np.any(pair_top5==true_pair[:,None],axis=1)
    out['error_category']=categories
    return stats,out

if RUN_FINAL_OFFICIAL_TEST:
    assert not OFFICIAL_DIR.exists() or not (OFFICIAL_DIR/'OFFICIAL_EVALUATION_COMPLETE.json').exists(), (
        'Official test is already completed. Do not repeat checkpoint selection or retest against it.')
    OFFICIAL_DIR.mkdir(parents=True,exist_ok=True)
    frozen_manifest={
        'stage':'FINAL_OFFICIAL_TEST','selection_policy':FINAL_SELECTION_POLICY,
        'crop_checkpoint':str(FINAL_CROP_CHECKPOINT),
        'disease_checkpoint':str(FINAL_DISEASE_CHECKPOINT),
        'crop_checkpoint_sha256':hashlib.sha256(FINAL_CROP_CHECKPOINT.read_bytes()).hexdigest(),
        'disease_checkpoint_sha256':hashlib.sha256(FINAL_DISEASE_CHECKPOINT.read_bytes()).hexdigest(),
        'candidate_count':9625,'test_seen_count':2000,'test_unseen_count':1448,
        'transform':'Resize256-CenterCrop224-ImageNetNormalize',
        'warning':'Official test results must not be used for hyperparameter / epoch / checkpoint selection'
    }
    manifest_file=OFFICIAL_DIR/'frozen_evaluation_protocol.json'
    if manifest_file.exists():
        assert json.loads(manifest_file.read_text())==frozen_manifest, 'Official protocol has changed: do not mix runs'
    else: manifest_file.write_text(json.dumps(frozen_manifest,indent=2))
    crop_model=load_frozen_model(FINAL_CROP_CHECKPOINT,'crop',0.3)
    disease_model=load_frozen_model(FINAL_DISEASE_CHECKPOINT,'disease',0.2)
    official_stats={}; official_predictions={}
    for split in ['seen','unseen']:
        stats,preds=evaluate_official_split(split,crop_model,disease_model)
        official_stats[split]=stats;official_predictions[split]=preds
        folder=OFFICIAL_DIR/'predictions';folder.mkdir(parents=True,exist_ok=True)
        preds.to_csv(folder/f'official_{split}_all_image_predictions.csv',index=False)
    del crop_model,disease_model
    gc.collect();torch.cuda.empty_cache()
    print('Official inference finished. Both prediction CSVs saved.')
else:
    print('Skipped final official inference (RUN_FINAL_OFFICIAL_TEST=False).')

## Cell 13 — Clean official metric tables and harmonic means

All percentages are labelled explicitly. The **primary B1 composition metric** is harmonic mean of official Seen and Unseen Composition Top-1. Also report independent Crop and Disease H, and Top-5 H for transparency.

In [ ]:
if RUN_FINAL_OFFICIAL_TEST:
    metric_rows=[]
    names=[('Crop','crop'),('Disease','disease'),('Composition','composition')]
    for task,prefix in names:
        for topk in [1,5]:
            key=f'{prefix}_top{topk}'
            s=official_stats['seen'][key];u=official_stats['unseen'][key]
            metric_rows.append({
                'Model / Task':task, 'Metric':f'Top-{topk} Accuracy',
                'Official Seen (%)':round(100*s,2),
                'Official Unseen (%)':round(100*u,2),
                'Seen–Unseen Harmonic Mean (%)':round(100*harmonic(s,u),2)
            })
    official_metric_table=pd.DataFrame(metric_rows)
    official_metric_table.to_csv(OFFICIAL_DIR/'01_official_seen_unseen_metrics.csv',index=False)
    (OFFICIAL_DIR/'01_official_seen_unseen_metrics.json').write_text(json.dumps({
        'seen':official_stats['seen'],'unseen':official_stats['unseen'],
        'primary_metric':'composition_top1_seen_unseen_harmonic_mean',
        'primary_harmonic_mean':harmonic(official_stats['seen']['composition_top1'],official_stats['unseen']['composition_top1'])
    },indent=2))
    print('FINAL OFFICIAL TEST — B1 C4 + D1 | 9,625 candidate compositions')
    display(official_metric_table)
    print('PRIMARY COMPOSITION H:',f'{100*harmonic(official_stats["seen"]["composition_top1"],official_stats["unseen"]["composition_top1"]):.2f}%')
else:print('Run official inference first.')

## Cell 14 — Confusion matrices: Crop, Disease, and Composition, for BOTH test splits

Save **full numerical CSV matrices**, plus readable heatmap images. For Crop/Disease, plot a readable top-20 confusion subset plus an `OTHER` prediction column; full 55×55 and 175×175 data remain in the CSV. For Composition, save a full matrix across all ground-truth classes in the split, plus a readable top-20 heatmap. `OTHER` captures predictions outside the displayed class subset (not an additional class).

In [ ]:
import matplotlib.pyplot as plt

def save_confusion_table_and_image(df,split,task,output_dir):
    if task=='crop':
        true=df['crop_id'].astype(int).to_numpy()
        pred=df['pred_crop_id'].astype(int).to_numpy()
        all_ids=crop_ids
        labels={int(row.crop_id):str(row.crop_name) for row in df.itertuples()}
        # Source train provides full class labels, not just those appearing in this split.
        labels.update({int(r.crop_id):str(r.crop_name) for r in full_train_df.itertuples()})
        lookup=lambda n:f'{n} | {labels.get(n,n)}'
    elif task=='disease':
        true=df['disease_id'].astype(int).to_numpy()
        pred=df['pred_disease_id'].astype(int).to_numpy()
        all_ids=disease_ids
        labels={int(r.disease_id):str(r.disease_name) for r in full_train_df.itertuples()}
        lookup=lambda n:f'{n} | {labels.get(n,n)}'
    else:
        true=list(zip(df.crop_id.astype(int),df.disease_id.astype(int)))
        pred=list(zip(df.pred_composition_crop_id.astype(int),df.pred_composition_disease_id.astype(int)))
        all_ids=sorted(set(true)|set(pred))
        cname={int(r.crop_id):str(r.crop_name) for r in full_train_df.itertuples()}
        dname={int(r.disease_id):str(r.disease_name) for r in full_train_df.itertuples()}
        lookup=lambda t:f'{cname.get(t[0],t[0])} + {dname.get(t[1],t[1])} [{t[0]}_{t[1]}]'
    if task!='composition':
        mat=confusion_matrix(true,pred,labels=all_ids)
        names=[lookup(x) for x in all_ids]
    else:
        # sklearn accepts 1D scalar keys only; encode pairs exactly as integers.
        enc=lambda a:int(a[0])*100000+int(a[1])
        ordered=sorted(all_ids)
        mat=confusion_matrix([enc(x) for x in true],[enc(x) for x in pred],labels=[enc(x) for x in ordered])
        names=[lookup(x) for x in ordered]
    assert int(mat.sum())==len(df)
    cm=pd.DataFrame(mat,index=names,columns=names)
    cm.to_csv(output_dir/f'{split}_{task}_full_confusion_matrix.csv',index_label='Actual class')
    # Choose most frequent true labels for a compact readable plot, aggregate all other predicted labels.
    selected=np.argsort(-mat.sum(axis=1))[:min(20,len(names))]
    selected=selected[mat[selected].sum(axis=1)>0]
    small=mat[np.ix_(selected,selected)]
    other=mat[selected].sum(axis=1)-small.sum(axis=1)
    compact=np.column_stack([small,other])
    display_names=[names[i] for i in selected]
    pd.DataFrame(compact,index=display_names,columns=display_names+['OTHER PREDICTIONS']).to_csv(
        output_dir/f'{split}_{task}_top20_readable_confusion.csv',index_label='Actual class')
    fig,ax=plt.subplots(figsize=(max(11,0.6*len(selected)+3),max(8,0.55*len(selected)+2)))
    im=ax.imshow(compact,cmap='Blues',aspect='auto')
    ax.set_xticks(range(compact.shape[1]),display_names+['OTHER'],rotation=90,fontsize=7)
    ax.set_yticks(range(len(selected)),display_names,fontsize=7)
    ax.set_title(f'B1 FINAL OFFICIAL {split.upper()} — {task.upper()} Confusion (Top-20 actual classes)')
    ax.set_xlabel('Predicted class');ax.set_ylabel('Actual class')
    fig.colorbar(im,ax=ax,label='Image count');fig.tight_layout()
    fig.savefig(output_dir/f'{split}_{task}_top20_confusion_matrix.png',dpi=180,bbox_inches='tight')
    plt.show();plt.close(fig)
    print(f'Saved {split} {task}: full {mat.shape[0]}×{mat.shape[1]} CSV, readable Top-20 image')

if RUN_FINAL_OFFICIAL_TEST:
    cm_dir=OFFICIAL_DIR/'confusion_matrices';cm_dir.mkdir(parents=True,exist_ok=True)
    for split in ['seen','unseen']:
        for task in ['crop','disease','composition']:
            save_confusion_table_and_image(official_predictions[split],split,task,cm_dir)
else:print('Run official inference first.')

## Cell 15 — Misclassification: separate Crop and Disease errors, four composition categories

Each split gets full CSVs for `crop_wrong`, `disease_wrong`, `both_correct`, `crop_wrong_only`, `disease_wrong_only`, `both_wrong` and counts. Galleries show **actual official test images** with true and predicted labels, not generated stock photos. Galleries are for final reporting **only**, not further optimization.

In [ ]:
def short_label(x,maxlen=40):
    x=str(x)
    return x if len(x)<=maxlen else x[:maxlen-3]+'...'

def image_gallery(df,save_path,title,max_images=16):
    if df.empty: return
    sampled=df.head(max_images)
    n=len(sampled);cols=4;rows=int(np.ceil(n/cols))
    fig,axes=plt.subplots(rows,cols,figsize=(16,rows*4.3))
    axes=np.asarray(axes).reshape(-1)
    for ax,(_,r) in zip(axes,sampled.iterrows()):
        with Image.open(image_lookup[r.filename]) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'TRUE: {short_label(r.crop_name,24)} / {short_label(r.disease_name,24)}\n'
                     f'PRED: {short_label(r.pred_crop_name,24)} / {short_label(r.pred_disease_name,24)}\n'
                     f'{short_label(r.filename,42)}',fontsize=8)
        ax.axis('off')
    for ax in axes[n:]:ax.axis('off')
    fig.suptitle(title,fontsize=14)
    fig.tight_layout(rect=(0,0,1,0.97));fig.savefig(save_path,dpi=170,bbox_inches='tight')
    plt.show();plt.close(fig)

if RUN_FINAL_OFFICIAL_TEST:
    # Human-readable predicted names (from the training metadata, not guessed from crop/disease IDs).
    crop_names={int(r.crop_id):str(r.crop_name) for r in full_train_df.itertuples()}
    disease_names={int(r.disease_id):str(r.disease_name) for r in full_train_df.itertuples()}
    count_rows=[]
    for split,df in official_predictions.items():
        df=df.copy()
        df['pred_crop_name']=df.pred_crop_id.map(crop_names)
        df['pred_disease_name']=df.pred_disease_id.map(disease_names)
        df['pred_composition_crop_name']=df.pred_composition_crop_id.map(crop_names)
        df['pred_composition_disease_name']=df.pred_composition_disease_id.map(disease_names)
        folder=OFFICIAL_DIR/'misclassification'/split
        folder.mkdir(parents=True,exist_ok=True)
        df.to_csv(folder/'all_official_samples_with_named_predictions.csv',index=False)
        selections={
            'crop_wrong':df[~df.crop_correct],
            'disease_wrong':df[~df.disease_correct],
            'both_correct':df[df.error_category=='both_correct'],
            'crop_wrong_only':df[df.error_category=='crop_wrong_only'],
            'disease_wrong_only':df[df.error_category=='disease_wrong_only'],
            'both_wrong':df[df.error_category=='both_wrong'],
        }
        print(f'\n{split.upper()} — Error categories and sample galleries')
        for name,subset in selections.items():
            subdir=folder/name;subdir.mkdir(parents=True,exist_ok=True)
            subset.to_csv(subdir/f'{name}_all_samples.csv',index=False)
            count_rows.append({'Split':split,'Category':name,'Count':len(subset),'Percentage (%)':round(100*len(subset)/len(df),2)})
            image_gallery(subset,subdir/f'{name}_gallery.png',f'OFFICIAL {split.upper()} — {name} | C4 + D1',MAX_GALLERY_PER_CATEGORY)
    counts=pd.DataFrame(count_rows)
    display(counts)
    counts.to_csv(OFFICIAL_DIR/'02_official_misclassification_category_counts.csv',index=False)
else:print('Run official inference first.')

## Cell 16 — Final audit and file index

This cell verifies every expected deliverable. If all checks pass, it writes a completion marker so that this Official Test is not accidentally rerun with different checkpoints. Never choose a new model based on the official results displayed here.

In [ ]:
if RUN_FINAL_OFFICIAL_TEST:
    required=[
        OFFICIAL_DIR/'frozen_evaluation_protocol.json',
        OFFICIAL_DIR/'01_official_seen_unseen_metrics.csv',
        OFFICIAL_DIR/'01_official_seen_unseen_metrics.json',
        OFFICIAL_DIR/'02_official_misclassification_category_counts.csv',
    ]
    for split in ['seen','unseen']:
        required.append(OFFICIAL_DIR/'predictions'/f'official_{split}_all_image_predictions.csv')
        required.append(OFFICIAL_DIR/'misclassification'/split/'all_official_samples_with_named_predictions.csv')
        for task in ['crop','disease','composition']:
            required.append(OFFICIAL_DIR/'confusion_matrices'/f'{split}_{task}_full_confusion_matrix.csv')
            required.append(OFFICIAL_DIR/'confusion_matrices'/f'{split}_{task}_top20_confusion_matrix.png')
    missing=[str(p) for p in required if not p.is_file()]
    assert not missing, f'Missing final deliverables: {missing}'
    marker=OFFICIAL_DIR/'OFFICIAL_EVALUATION_COMPLETE.json'
    marker.write_text(json.dumps({'status':'COMPLETE','selection_policy':FINAL_SELECTION_POLICY,
                                  'checkpoint_crop':str(FINAL_CROP_CHECKPOINT),
                                  'checkpoint_disease':str(FINAL_DISEASE_CHECKPOINT),
                                  'seen_samples':2000,'unseen_samples':1448,
                                  'candidate_compositions':9625},indent=2))
    index=sorted(str(p.relative_to(OFFICIAL_DIR)) for p in OFFICIAL_DIR.rglob('*') if p.is_file())
    (OFFICIAL_DIR/'FILE_INDEX.txt').write_text('\n'.join(index)+'\n')
    print('FINAL OFFICIAL EVALUATION COMPLETE')
    print('All outputs saved:',OFFICIAL_DIR)
    print('Files:',len(index)+1)
    display(official_metric_table)
else:
    print('Official evaluation remains DISABLED. Fine-tuning files are unchanged.')